# Bigdata Research Tools - Complete Examples

This notebook demonstrates all the key functionality of the `bigdata-research-tools` library with practical, working examples.

## Table of Contents

1. [Setup and Authentication](#1-setup-and-authentication)
2. [Basic Search Examples](#2-basic-search-examples)
   - [Search by Companies](#search-by-companies)
   - [Custom Query Search](#custom-query-search)
3. [Advanced Workflows](#3-advanced-workflows)
   - [Narrative Miner](#narrative-miner)
   - [Thematic Screener](#thematic-screener)
   - [Risk Analyzer](#risk-analyzer)
4. [Utility Tools](#4-utility-tools)
   - [Query Builder](#query-builder)
   - [Portfolio Constructor](#portfolio-constructor)
5. [Tips and Best Practices](#5-tips-and-best-practices)

---


## 1. Setup and Authentication

First, let's import the necessary libraries and set up authentication.


In [ ]:
# Import core libraries
import logging
import pandas as pd
import numpy as np
from typing import List, Dict
from dotenv import load_dotenv

# Bigdata client imports
from bigdata_client.models.search import DocumentType, SortBy
from bigdata_client.daterange import AbsoluteDateRange

# Bigdata research tools imports
from bigdata_research_tools.client import bigdata_connection
from bigdata_research_tools.search.screener_search import search_by_companies
from bigdata_research_tools.search.search import run_search
from bigdata_research_tools.search.query_builder import (
    build_batched_query,
    EntitiesToSearch,
    create_date_ranges
)
from bigdata_research_tools.workflows import NarrativeMiner, ThematicScreener
from bigdata_research_tools.workflows.risk_analyzer import RiskAnalyzer
from bigdata_research_tools.portfolio.portfolio_constructor import (
    PortfolioConstructor, WeightMethod
)

# Configure clean logging for notebook
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully")


In [ ]:
# Load environment variables and connect to Bigdata API
load_dotenv()

# Initialize connection
bigdata = bigdata_connection()

print("✅ Connected to Bigdata API")
print("📋 Ready to run examples!")


---

## 2. Basic Search Examples

Let's start with the fundamental search capabilities.

### Search by Companies

This example shows how to search for documents mentioning specific companies and topics.


In [ ]:
def demo_search_by_companies():
    """Demonstrate search_by_companies functionality."""
    
    print("🔍 Search by Companies Example")
    print("=" * 40)
    
    # Get companies using ticker symbols (reliable method)
    tickers = ["AAPL", "MSFT", "TSLA"]
    companies = []
    
    print("Finding companies using tickers...")
    for ticker in tickers:
        results = bigdata.knowledge_graph.autosuggest(ticker, limit=1)
        if results:
            companies.extend(results)
            print(f"  ✅ Found: {results[0].name} ({ticker})")
    
    if not companies:
        print("❌ No companies found")
        return None
    
    # Define search sentences
    sentences = ["AI is transforming business", "Cloud adoption is accelerating"]
    
    print(f"\n📊 Searching for content across {len(companies)} companies...")
    
    try:
        # Search for documents
        results_df = search_by_companies(
            companies=companies,
            sentences=sentences,
            start_date="2024-01-01",
            end_date="2024-06-30",
            scope=DocumentType.NEWS,
            document_limit=20,
            batch_size=5
        )
        
        # Display results
        if not results_df.empty:
            print(f"\n✅ Found {len(results_df)} relevant documents")
            
            # Company breakdown
            company_counts = results_df['entity_name'].value_counts()
            print("\n📈 Documents by company:")
            for company, count in company_counts.items():
                print(f"  {company}: {count} documents")
            
            # Sample headlines
            print("\n📰 Sample headlines:")
            for headline in results_df['headline'].head(3):
                print(f"  • {headline}")
            
            return results_df
        else:
            print("⚠️ No documents found")
            return None
            
    except ValueError as e:
        if "No rows to process" in str(e):
            print("⚠️ No documents found matching the search criteria")
            print("💡 Try different date ranges or search terms")
            return None
        else:
            raise

# Run the example
search_results = demo_search_by_companies()


In [ ]:
# Display the results DataFrame if we got results
if search_results is not None:
    print("📊 Results DataFrame:")
    display(search_results.head(10))
else:
    print("No results to display")


### Custom Query Search

This example demonstrates building custom queries and using the run_search function.


In [ ]:
def demo_run_search():
    """Demonstrate run_search with custom query building."""
    
    print("🔧 Custom Query Search Example")
    print("=" * 40)
    
    # Define entities to search for
    entities = EntitiesToSearch(
        companies=["Apple Inc", "Google", "Microsoft Corp"],
        topic=["earnings", "financial results"],
        concepts=["revenue growth", "profit margins"]
    )
    
    # Define search sentences
    sentences = [
        "quarterly earnings performance",
        "revenue growth and profitability"
    ]
    
    print("🔨 Building search queries...")
    
    # Build queries
    queries = build_batched_query(
        sentences=sentences,
        keywords=["earnings", "revenue", "profit"],
        entities=entities,
        control_entities=None,
        sources=None,
        batch_size=5,
        fiscal_year=None,
        scope=DocumentType.NEWS,
        custom_batches=None
    )
    
    print(f"✅ Generated {len(queries)} search queries")
    
    # Create date ranges
    date_ranges = create_date_ranges("2024-10-01", "2024-12-31", "M")
    print(f"📅 Searching across {len(date_ranges)} time periods")
    
    # Execute search
    print("🔍 Executing search...")
    
    search_results = run_search(
        queries=queries,
        date_ranges=date_ranges,
        scope=DocumentType.NEWS,
        limit=8,
        only_results=True
    )
    
    # Process results
    all_documents = []
    
    for result_batch in search_results:
        for doc in result_batch:
            # Convert timezone-aware datetime for compatibility
            timestamp_naive = doc.timestamp.replace(tzinfo=None) if doc.timestamp else None
            
            doc_data = {
                'timestamp': timestamp_naive,
                'headline': doc.headline,
                'source': doc.source.name if doc.source else 'Unknown',
                'doc_id': doc.id if hasattr(doc, 'id') else 'N/A'
            }
            all_documents.append(doc_data)
    
    # Convert to DataFrame
    results_df = pd.DataFrame(all_documents)
    
    if not results_df.empty:
        print(f"\n✅ Found {len(results_df)} documents total")
        
        # Source distribution
        source_counts = results_df['source'].value_counts()
        print("\n📊 Documents by source:")
        for source, count in source_counts.head(5).items():
            print(f"  {source}: {count} documents")
        
        # Sample headlines
        print("\n📰 Sample headlines:")
        for headline in results_df['headline'].head(3):
            print(f"  • {headline}")
        
        return results_df
    else:
        print("⚠️ No documents found")
        return None

# Run the example
custom_search_results = demo_run_search()


---

## 3. Portfolio Constructor Example

Build balanced and weighted portfolios with sophisticated constraint management.


In [ ]:
def demo_portfolio_constructor():
    """Demonstrate portfolio construction with different weighting methods."""
    
    print("💼 Portfolio Constructor Example")
    print("=" * 40)
    
    # Create sample data
    print("📊 Creating sample company data...")
    
    np.random.seed(42)  # For reproducible results
    
    sectors = ['Technology', 'Healthcare', 'Financial Services', 'Consumer Goods', 'Energy']
    industries = ['Software', 'Pharmaceuticals', 'Banking', 'Retail', 'Oil & Gas']
    
    n_companies = 30
    data = {
        'Company': [f'Company_{i:03d}' for i in range(1, n_companies + 1)],
        'Sector': np.random.choice(sectors, n_companies),
        'Industry': np.random.choice(industries, n_companies),
        'Market_Cap': np.random.lognormal(mean=2, sigma=1.5, size=n_companies),
        'Composite_Score': np.random.normal(loc=75, scale=15, size=n_companies),
        'ESG_Score': np.random.uniform(20, 95, n_companies),
    }
    
    df = pd.DataFrame(data)
    
    # Clean data
    df['Composite_Score'] = np.clip(df['Composite_Score'], 0, 100)
    df['ESG_Score'] = np.round(df['ESG_Score'], 1)
    df['Market_Cap'] = np.round(df['Market_Cap'], 2)
    
    print(f"  ✅ Created dataset with {len(df)} companies across {len(sectors)} sectors")
    
    # Initialize portfolio constructor
    constructor = PortfolioConstructor(
        max_iterations=1000,
        tolerance=1e-6
    )
    
    portfolios = {}
    
    # Example 1: Equal-weighted portfolio
    print("\n📝 Example 1: Equal-Weighted Portfolio")
    
    portfolio_equal = constructor.construct_portfolio(
        df=df,
        score_col="Composite_Score",
        balance_col="Sector",
        size=15,
        max_position_weight=0.08,
        max_category_weight=0.25,
        weight_method=WeightMethod.EQUAL
    )
    
    print(f"  ✅ Portfolio Size: {len(portfolio_equal)} companies")
    print(f"  📊 Sectors: {portfolio_equal['Sector'].nunique()}")
    print(f"  ⚖️ Weight Range: {portfolio_equal['weight'].min():.1%} - {portfolio_equal['weight'].max():.1%}")
    
    portfolios['equal_weighted'] = portfolio_equal
    
    # Example 2: Market cap weighted portfolio
    print("\n📝 Example 2: Market Cap Weighted Portfolio")
    
    portfolio_mcap = constructor.construct_portfolio(
        df=df,
        score_col="Composite_Score",
        balance_col="Sector",
        weight_col="Market_Cap",
        size=15,
        max_position_weight=0.10,
        max_category_weight=0.30,
        weight_method=WeightMethod.COLUMN
    )
    
    print(f"  ✅ Portfolio Size: {len(portfolio_mcap)} companies")
    print(f"  📊 Sectors: {portfolio_mcap['Sector'].nunique()}")
    print(f"  ⚖️ Weight Range: {portfolio_mcap['weight'].min():.1%} - {portfolio_mcap['weight'].max():.1%}")
    
    portfolios['mcap_weighted'] = portfolio_mcap
    
    return portfolios, df

# Run the portfolio construction examples
portfolios, sample_data = demo_portfolio_constructor()


In [ ]:
# Display portfolio comparison
print("📊 Portfolio Comparison:")
print("=" * 50)

for name, portfolio in portfolios.items():
    print(f"\n{name.replace('_', ' ').title()}:")
    print(f"  Companies: {len(portfolio)}")
    print(f"  Total Weight: {portfolio['weight'].sum():.1%}")
    
    # Sector allocation
    sector_weights = portfolio.groupby('Sector')['weight'].sum().sort_values(ascending=False)
    print("  Top Sectors:")
    for sector, weight in sector_weights.head(3).items():
        print(f"    {sector}: {weight:.1%}")

# Display sample of equal-weighted portfolio
print("\n📋 Sample Equal-Weighted Portfolio:")
display(portfolios['equal_weighted'][['Company', 'Sector', 'Composite_Score', 'weight']].head(10))


---

## 4. Tips and Best Practices

Here are key tips for using the library effectively.


In [ ]:
print("🚀 Performance Optimization Tips")
print("=" * 40)

print("""
1. 📊 Batch Size Optimization:
   • Small datasets: batch_size=2-5
   • Large datasets: batch_size=10-25
   • Very large: batch_size=50+

2. 📅 Date Range Management:
   • Use shorter date ranges for faster results
   • Recent data (last 6 months) is more available
   • Quarterly intervals (3M) balance detail vs speed

3. 🎯 Document Limits:
   • Start with document_limit=10-20 for testing
   • Increase to 50+ for comprehensive analysis
   • Monitor API rate limits

4. 🔍 Search Strategy:
   • Use autosuggest() for company lookup (more reliable)
   • Start with sentences=None for broader results
   • Add specific sentences/keywords to narrow focus

5. 💾 Data Management:
   • Convert timezone-aware dates before Excel export
   • Cache company lookups to avoid repeated API calls
   • Use try-catch for graceful error handling
""")


## Summary

This notebook has demonstrated the key capabilities of the Bigdata Research Tools library:

✅ **Basic Search Functions**: `search_by_companies()` and `run_search()`  
✅ **Portfolio Constructor**: Build balanced portfolios with constraints  
✅ **Best Practices**: Performance optimization and error handling  

### Next Steps

1. **Customize Examples**: Modify the company lists, date ranges, and search terms for your specific use case
2. **Combine Workflows**: Use multiple tools together for comprehensive analysis
3. **Scale Up**: Increase batch sizes and document limits for production use
4. **Export Results**: Save DataFrames to Excel/CSV for further analysis

### Additional Workflows

The library also includes advanced AI-powered workflows:
- **NarrativeMiner**: Track narrative evolution over time
- **ThematicScreener**: Analyze company exposure to themes
- **RiskAnalyzer**: Assess risk exposure with taxonomies

*Note: These require LLM access (OpenAI API key) and may take longer to run.*

For more detailed documentation, see the [USER_GUIDE.md](USER_GUIDE.md) file.
